# Scientific Paper Domain Classification

**Course:** CSE440: Natural Language Processing II - Lab Project, Summer 2026

**Group number, section, members, and IDs:** Pending

Input: an English scientific-paper abstract from X.txt.  
Target: one of seven parent domains from YL1.txt.


# 1. Problem Definition and Research Questions

The system predicts Computer Science, Electrical Engineering, Psychology, Mechanical Engineering, Civil Engineering, Medical Science, or Biochemistry. It may support literature organization, search, recommendation, reviewer routing, and indexing audits. Interdisciplinary papers remain a limitation because the dataset assigns one parent label.

**RQ1:** Which TF-IDF traditional classifier performs best?  
**RQ2:** How do recurrent cell type and bidirectionality affect macro-F1?  
**RQ3:** Does BERT Base justify its additional computational cost?  
**RQ4:** Which domains are confused, and how do dataset properties explain the errors?


# 2. Colab Setup and Reproducibility

Mount Drive first. All large data and generated artifacts stay under one configurable Drive directory.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")
print("Google Drive mounted successfully.")


## Install Missing Libraries


In [ ]:
import importlib.util
import subprocess
import sys

packages = {
    "langdetect": "langdetect==1.0.9",
    "gensim": "gensim>=4.3,<5",
    "wordcloud": "wordcloud>=1.9,<2",
    "transformers": "transformers>=4.46,<5",
    "datasets": "datasets>=3.1,<5",
    "accelerate": "accelerate>=1.1,<2",
}
missing = [requirement for module, requirement in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
print(f"Dependency check complete. Installed this session: {missing or 'none'}")


## Libraries


In [ ]:
from pathlib import Path
from importlib.metadata import PackageNotFoundError, version
import hashlib
import json
import platform
import random
import re
import sys
import unicodedata

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psutil
import seaborn as sns
import torch
from IPython.display import display
from langdetect import DetectorFactory, detect_langs
from sklearn.model_selection import StratifiedGroupKFold, train_test_split
from wordcloud import STOPWORDS, WordCloud

print("Libraries imported successfully.")


## Project Paths and Random Seed


In [ ]:
SEED = 42
PROJECT_DRIVE_DIR = Path("/content/drive/MyDrive/CSE440_Project")
DATA_DIR = PROJECT_DRIVE_DIR / "data" / "WOS-11967"
RESULTS_DIR = PROJECT_DRIVE_DIR / "results"
FIGURES_DIR = RESULTS_DIR / "figures"
HISTORIES_DIR = RESULTS_DIR / "histories"
CHECKPOINTS_DIR = RESULTS_DIR / "checkpoints"
for path in [DATA_DIR, RESULTS_DIR, FIGURES_DIR, HISTORIES_DIR, CHECKPOINTS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DetectorFactory.seed = SEED

print("Project directory:", PROJECT_DRIVE_DIR)
print("Random seed:", SEED)


## Environment Information


In [ ]:
names = ["numpy", "pandas", "scikit-learn", "torch", "transformers"]
versions = {}
for name in names:
    try:
        versions[name] = version(name)
    except PackageNotFoundError:
        versions[name] = "not installed"

environment = pd.DataFrame({
    "item": ["Python", "Platform", "GPU available", "GPU", "RAM GiB", "Drive directory"],
    "value": [
        sys.version.split()[0],
        platform.platform(),
        torch.cuda.is_available(),
        torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU runtime",
        round(psutil.virtual_memory().total / 1024**3, 2),
        str(PROJECT_DRIVE_DIR),
    ],
})
display(environment)
display(pd.DataFrame(versions.items(), columns=["package", "version"]))


# 3. Dataset Provenance and Loading

**Dataset:** Web of Science WOS-11967, version 6  
**Source:** https://data.mendeley.com/datasets/9rw3vkcfy4/6  
**DOI:** 10.17632/9rw3vkcfy4/6  
**Licence:** CC BY 4.0  
**Associated work:** HDLTex: Hierarchical Deep Learning for Text Classification

X.txt is the only model input. YL1.txt is the only target. Y.txt and YL2.txt are audit-only files.


## Verify Dataset Files


In [ ]:
def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with path.open("rb") as source:
        for chunk in iter(lambda: source.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


files = {
    "abstracts": DATA_DIR / "X.txt",
    "parent_labels": DATA_DIR / "YL1.txt",
    "global_child_labels": DATA_DIR / "Y.txt",
    "within_parent_child_labels": DATA_DIR / "YL2.txt",
}
file_audit = pd.DataFrame([
    {
        "role": role,
        "file": path.name,
        "exists": path.is_file(),
        "size_bytes": path.stat().st_size if path.is_file() else None,
        "sha256": sha256_file(path) if path.is_file() else None,
    }
    for role, path in files.items()
])
display(file_audit)
missing_required = [
    files[role].name for role in ["abstracts", "parent_labels"]
    if not files[role].is_file()
]
if missing_required:
    raise FileNotFoundError(f"Place {missing_required} in {DATA_DIR}")
print("Required source files are present and hashed.")


## Read Abstracts and Parent Labels


In [ ]:
def read_lines(path):
    return path.read_text(encoding="utf-8-sig").splitlines()


abstracts = read_lines(files["abstracts"])
raw_labels = read_lines(files["parent_labels"])
if len(abstracts) != len(raw_labels):
    raise ValueError("X.txt and YL1.txt are not aligned.")

numeric_labels = pd.to_numeric(pd.Series(raw_labels), errors="coerce")
if numeric_labels.isna().any():
    raise ValueError("YL1.txt contains malformed labels.")

print("Abstract and label rows are aligned.")


## Create the Dataset


In [ ]:
label_mapping = {
    0: "Computer Science",
    1: "Electrical Engineering",
    2: "Psychology",
    3: "Mechanical Engineering",
    4: "Civil Engineering",
    5: "Medical Science",
    6: "Biochemistry",
}
labels = numeric_labels.astype(int)
unexpected = sorted(set(labels) - set(label_mapping))
if unexpected:
    raise ValueError(f"Unexpected parent labels: {unexpected}")

papers = pd.DataFrame({
    "source_row": np.arange(len(abstracts)),
    "document_id": [f"wos11967_v6_{row:05d}" for row in range(len(abstracts))],
    "text": abstracts,
    "label_id": labels,
})
papers["label_name"] = papers["label_id"].map(label_mapping)
display(papers[["document_id", "label_id", "label_name"]].head())
display(papers.groupby(["label_id", "label_name"], as_index=False).size())
print(f"Aligned modeling table created with {len(papers):,} rows.")


In [ ]:
print("Dataset shape:", papers.shape)
print("Number of classes:", papers["label_id"].nunique())
display(papers["label_name"].value_counts().rename_axis("Domain").reset_index(name="Samples"))


# 4. Data-Quality Audit

Measure issues before treatment. Do not silently remove questionable rows. Duplicate groups with conflicting labels block the split.


## Missing Values and Duplicate Abstracts


In [ ]:
def normalize_text(text):
    text = unicodedata.normalize("NFKC", text)
    return re.sub(r"\s+", " ", text.lower()).strip()


papers["normalized_text"] = papers["text"].map(normalize_text)
papers["raw_sha256"] = papers["text"].map(
    lambda value: hashlib.sha256(value.encode("utf-8")).hexdigest()
)
papers["normalized_sha256"] = papers["normalized_text"].map(
    lambda value: hashlib.sha256(value.encode("utf-8")).hexdigest()
)
papers["word_count"] = papers["text"].str.split().str.len()
papers["character_count"] = papers["text"].str.len()

exact_duplicate_rows = int(papers.duplicated("raw_sha256", keep=False).sum())
normalized_duplicate_rows = int(papers.duplicated("normalized_sha256", keep=False).sum())
exact_conflicts = int((papers.groupby("raw_sha256")["label_id"].nunique() > 1).sum())
normalized_conflicts = int(
    (papers.groupby("normalized_sha256")["label_id"].nunique() > 1).sum()
)
quality = pd.DataFrame({
    "check": [
        "Rows", "Empty abstracts", "Missing labels", "Exact duplicate rows",
        "Normalized duplicate rows", "Exact conflicting groups",
        "Normalized conflicting groups",
    ],
    "count": [
        len(papers), int(papers["normalized_text"].eq("").sum()),
        int(papers["label_id"].isna().sum()), exact_duplicate_rows,
        normalized_duplicate_rows, exact_conflicts, normalized_conflicts,
    ],
})
display(quality)
if exact_conflicts or normalized_conflicts:
    raise ValueError("Conflicting duplicate labels require manual review.")
print("Missing-value and duplicate audit complete.")


## Abstract Length and Formatting


In [ ]:
control_pattern = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f]")
formatting = pd.DataFrame({
    "check": ["Control characters", "HTML-like markup", "Very short texts", "Very long texts"],
    "count": [
        int(papers["text"].map(lambda value: bool(control_pattern.search(value))).sum()),
        int(papers["text"].str.contains(r"<[^>]+>", regex=True, na=False).sum()),
        int((papers["word_count"] < papers["word_count"].quantile(0.01)).sum()),
        int((papers["word_count"] > papers["word_count"].quantile(0.99)).sum()),
    ],
})
display(formatting)
display(papers.nlargest(5, "word_count")[
    ["document_id", "label_name", "word_count", "character_count"]
])
print("Formatting and length-outlier audit complete.")


# 5. Language and Child-Label Audit

Automatic language detection is only supporting evidence. A fixed stratified sample is also displayed for manual inspection. Child labels never enter the modeling dataframe.


## Child-Label Discrepancy


In [ ]:
child_rows = []
for role in ["global_child_labels", "within_parent_child_labels"]:
    path = files[role]
    values = read_lines(path) if path.is_file() else []
    child_rows.append({
        "role": role,
        "file_present": path.is_file(),
        "rows": len(values) if values else None,
        "unique_values": pd.Series(values).nunique() if values else None,
        "used_as_feature": False,
    })
display(pd.DataFrame(child_rows))

print("Child labels were audited but not added as features.")


## English-Language Check


In [ ]:
language_sample = pd.concat([
    group.sample(n=min(5, len(group)), random_state=SEED)
    for _, group in papers.groupby("label_id", sort=True)
], ignore_index=True)

def language_result(text):
    try:
        result = detect_langs(text)[0]
        return pd.Series([result.lang, result.prob])
    except Exception:
        return pd.Series(["undetermined", np.nan])


language_sample[["detected_language", "probability"]] = (
    language_sample["text"].apply(language_result)
)
language_sample["excerpt"] = language_sample["text"].str.slice(0, 300)
display(language_sample.groupby(
    ["label_name", "detected_language"], as_index=False
).size())
display(language_sample[
    ["document_id", "label_name", "detected_language", "probability", "excerpt"]
])
print(f"Language audit displayed {len(language_sample)} fixed stratified samples.")


# 6. Exploratory Data Analysis

These are descriptive summaries, not model-fitting decisions. Word clouds supplement rather than replace quantitative analysis.


## Class Distribution


In [ ]:
class_distribution = (
    papers.groupby(["label_id", "label_name"], as_index=False)
    .size().rename(columns={"size": "samples"})
)
class_distribution["percentage"] = (
    100 * class_distribution["samples"] / len(papers)
)
imbalance_ratio = class_distribution["samples"].max() / class_distribution["samples"].min()
display(class_distribution)

plt.figure(figsize=(10, 5))
sns.barplot(data=class_distribution, x="samples", y="label_name",
            hue="label_name", legend=False, palette="Set2")
plt.title("WOS-11967 Parent-Domain Distribution")
plt.xlabel("Abstracts")
plt.ylabel("")
plt.tight_layout()
class_figure = FIGURES_DIR / "class_distribution.png"
plt.savefig(class_figure, dpi=200, bbox_inches="tight")
plt.show()


## Abstract Length Statistics


In [ ]:
length_summary = papers[["word_count", "character_count"]].describe(
    percentiles=[0.25, 0.5, 0.75, 0.95, 0.99]
).T
display(length_summary.round(2))
display(papers.groupby("label_name")["word_count"].agg(
    ["count", "mean", "median", "min", "max"]
).round(2))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(papers["word_count"], bins=50, ax=axes[0], color="#287271")
axes[0].set_title("Overall Abstract Word Counts")
sns.boxplot(data=papers, x="word_count", y="label_name",
            hue="label_name", legend=False, palette="Set2", ax=axes[1])
axes[1].set_title("Word Counts by Domain")
axes[1].set_ylabel("")
plt.tight_layout()
length_figure = FIGURES_DIR / "text_length_analysis.png"
plt.savefig(length_figure, dpi=200, bbox_inches="tight")
plt.show()
print(f"Imbalance ratio: {imbalance_ratio:.2f}:1. Figures saved in {FIGURES_DIR}")


## Word Clouds


In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(16, 20))
axes = axes.flatten()
for axis, (label_id, label_name) in zip(axes, label_mapping.items()):
    text = " ".join(papers.loc[papers["label_id"].eq(label_id), "text"])
    cloud = WordCloud(
        width=900, height=450, background_color="white", stopwords=STOPWORDS,
        max_words=100, random_state=SEED, collocations=False,
    ).generate(text)
    axis.imshow(cloud, interpolation="bilinear")
    axis.set_title(label_name)
    axis.axis("off")
axes[-1].axis("off")
plt.tight_layout()
wordcloud_figure = FIGURES_DIR / "class_wordclouds.png"
plt.savefig(wordcloud_figure, dpi=180, bbox_inches="tight")
plt.show()


## Representative Abstracts


In [ ]:
examples = pd.concat([
    group.sample(n=min(2, len(group)), random_state=SEED)
    for _, group in papers.groupby("label_id", sort=True)
], ignore_index=True)
examples["excerpt"] = examples["text"].str.slice(0, 350)
display(examples[["document_id", "label_name", "word_count", "excerpt"]])
print(f"Word clouds saved to {wordcloud_figure}; two examples displayed per class.")


# 7. Model-Specific Preprocessing Decisions

**TF-IDF:** safe Unicode/whitespace normalization and lowercasing; preserve scientific word forms; fit training text only.

**GloVe recurrent models:** minimal normalization; training-only vocabulary and sequence length; explicit padding and unknown tokens; report coverage and OOV rate.

**BERT:** official bert-base-uncased tokenizer with minimal normalization; preserve punctuation, numbers, abbreviations, and scientific symbols; choose maximum length from training WordPiece statistics.


# 8. Frozen Train/Validation/Test Split

Use 70% training, 15% validation, and 15% test with seed 42. Normalized duplicate groups, if present, must remain within one split. Every later model reloads the saved manifest.


## Create and Save the Split


In [ ]:
papers["duplicate_group_id"] = "group_" + papers["normalized_sha256"].str[:16]
if normalized_duplicate_rows:
    splitter = StratifiedGroupKFold(n_splits=20, shuffle=True, random_state=SEED)
    folds = np.full(len(papers), -1, dtype=int)
    for fold, (_, held_out) in enumerate(splitter.split(
        papers, papers["label_id"], groups=papers["duplicate_group_id"]
    )):
        folds[held_out] = fold
    splits = np.where(folds < 14, "train",
                      np.where(folds < 17, "validation", "test"))
    method = "20-fold stratified group assignment"
else:
    train_idx, temp_idx = train_test_split(
        papers.index, test_size=0.30, random_state=SEED,
        stratify=papers["label_id"],
    )
    val_idx, test_idx = train_test_split(
        temp_idx, test_size=0.50, random_state=SEED,
        stratify=papers.loc[temp_idx, "label_id"],
    )
    splits = np.full(len(papers), "", dtype=object)
    splits[train_idx], splits[val_idx], splits[test_idx] = "train", "validation", "test"
    folds = np.full(len(papers), -1, dtype=int)
    method = "two-stage stratified split"

papers["split"] = splits
papers["fold_id"] = folds
manifest = papers[[
    "document_id", "source_row", "label_id", "label_name",
    "duplicate_group_id", "normalized_sha256", "fold_id", "split",
]].copy()
manifest_path = RESULTS_DIR / "split_manifest.csv"
manifest.to_csv(manifest_path, index=False)
display(manifest.groupby(["label_name", "split"]).size().unstack(fill_value=0))
display(manifest["split"].value_counts().rename_axis("split").reset_index(name="samples"))
print(f"Frozen split saved with {method}: {manifest_path}")


## Validate the Split


In [ ]:
manifest_hash = sha256_file(manifest_path)
sets = {name: set(group["document_id"]) for name, group in manifest.groupby("split")}
checks = pd.DataFrame({
    "check": [
        "Every document assigned once",
        "No document overlap",
        "No duplicate-group overlap",
        "All seven classes in every split",
        "Manifest hash reload matches",
        "No conflicting duplicate labels",
    ],
    "passed": [
        len(manifest) == len(papers) and manifest["document_id"].is_unique,
        sets["train"].isdisjoint(sets["validation"])
        and sets["train"].isdisjoint(sets["test"])
        and sets["validation"].isdisjoint(sets["test"]),
        manifest.groupby("duplicate_group_id")["split"].nunique().max() == 1,
        manifest.groupby("split")["label_id"].nunique().eq(7).all(),
        sha256_file(manifest_path) == manifest_hash,
        exact_conflicts == 0 and normalized_conflicts == 0,
    ],
})
metadata = {
    "version": "wos11967_v6_seed42_v1",
    "seed": SEED,
    "method": method,
    "manifest_sha256": manifest_hash,
    "rows": len(manifest),
    "source_sha256": {
        row["file"]: row["sha256"]
        for _, row in file_audit[file_audit["exists"]].iterrows()
    },
}
metadata_path = RESULTS_DIR / "split_manifest_metadata.json"
metadata_path.write_text(json.dumps(metadata, indent=2, sort_keys=True), encoding="utf-8")
display(checks)
print(f"Split metadata saved to {metadata_path}")


## Pre-Training Validation Gate


In [ ]:
MILESTONE_REVIEW_APPROVED = False
gate = pd.DataFrame({
    "requirement": [
        "Source files verified", "Rows and labels aligned", "No empty abstracts",
        "No conflicting duplicates", "Language evidence displayed",
        "EDA figures produced", "Split integrity passed", "Team review approved",
    ],
    "passed": [
        not missing_required, len(abstracts) == len(raw_labels),
        papers["normalized_text"].ne("").all(),
        exact_conflicts == 0 and normalized_conflicts == 0,
        len(language_sample) > 0,
        class_figure.is_file() and length_figure.is_file() and wordcloud_figure.is_file(),
        checks["passed"].all(), MILESTONE_REVIEW_APPROVED,
    ],
})
display(gate)
status = "PASS" if gate["passed"].all() else "BLOCKED"
print(f"PRE-TRAINING GATE: {status}. Review pending items before Section 9.")


**Mandatory review stop:** Do not fit TF-IDF, load GloVe, construct models, or train until Sections 1-8 execute successfully and MILESTONE_REVIEW_APPROVED is deliberately set to True.


# 9. TF-IDF and Traditional Models

After approval, fit one training-only TF-IDF vectorizer: word unigrams/bigrams, min_df 2, max_df 0.95, at most 50,000 features, and sublinear term frequency.

Manual configurations: Logistic Regression C = 0.5, 1.0, 2.0; MultinomialNB alpha = 0.1, 0.5, 1.0; Random Forest = 200 trees/depth 50, 400/depth 100, and 500/unlimited.


# 10. GloVe and Sequence Preparation

Build the training-only vocabulary, choose sequence length from training statistics, load the team-provided GloVe 100d file from Drive, and report coverage and OOV rate.


# 11. Recurrent Models

Train SimpleRNN, GRU, LSTM, and their bidirectional variants with three shared profiles. Use batch size 64, at most 15 epochs, patience 2, gradient clipping, and the best validation macro-F1 checkpoint.


# 12. BERT Base

Analyze training WordPiece lengths, choose 256 or 384 tokens without test evidence, and tune bert-base-uncased at 1e-5, 2e-5, and 3e-5.


# 13. Validation-Based Selection

Save at least 30 completed official runs to tuning_results.csv: 9 traditional, 18 recurrent, and 3 BERT. Select by validation macro-F1, with accuracy as tie-breaker.


# 14. Final Test Evaluation

Evaluate one locked configuration per model. Report accuracy, macro-F1, weighted-F1, per-class metrics, full classification reports, and raw and normalized confusion matrices.


# 15. Comparison, Error Analysis, and Conclusions

Compare performance and cost, identify the best and worst models, analyze domain confusions, answer RQ1-RQ4, and discuss imbalance, interdisciplinary labels, truncation, and computational limits.
